In [1]:
!pip install torchmetrics --quiet

In [2]:
import pickle

import numpy as np
import pandas as pd

import torch
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import Dataset, DataLoader

from torchmetrics import Accuracy

In [3]:
class CustomDataset(Dataset):
    def __init__(self, dataset_path: str) -> None:
        data = pd.read_parquet(dataset_path)

        self.nrof_emb_categories = {
            "workclass_cat": 9,
            "education_cat": 16,
            "marital-status_cat": 7,
            "occupation_cat": 16,
            "relationship_cat": 6,
            "race_cat": 5,
            "sex_cat": 2,
            "native-country_cat": 42,
        }
        self.embedding_columns = list(self.nrof_emb_categories.keys())
        self.numeric_columns = [
            "age",
            "fnlwgt",
            "education-num",
            "capital-gain",
            "capital-loss",
            "hours-per-week",
        ]
        self.columns = self.embedding_columns + self.numeric_columns
        self.X = data[self.columns].reset_index(drop=True)
        self.y = np.asarray([0 if el == "<50k" else 1 for el in data["salary"].values], dtype=np.int32)

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx):
        row = self.X.take([idx], axis=0)
        row = {col: torch.tensor(row[col].values, dtype=torch.float32) for col in self.columns}
        return row, np.float32(self.y[idx])

In [4]:
class DenseFeatureLayer(torch.nn.Module):
    def __init__(
        self,
        nrof_cat: int,
        emb_dim: int,
        emb_columns: list[str],
        numeric_columns: list[str],
    ) -> None:
        super().__init__()

        self.emb_columns = emb_columns
        self.numeric_columns = numeric_columns
        self.numeric_feature_bn = torch.nn.BatchNorm1d(len(numeric_columns))

        input_size = len(emb_columns) + len(numeric_columns)
        self.first_feature_bn = torch.nn.BatchNorm1d(input_size)
        self.second_feature_bn = torch.nn.BatchNorm1d(input_size)

        # first order feature interactions
        self.first_order_embd = torch.nn.ModuleDict()
        for col in self.emb_columns:
            self.first_order_embd[col] = torch.nn.Embedding(nrof_cat[col], 1)
        self.first_order_scalar = torch.nn.ParameterDict({})
        for col in numeric_columns:
            self.first_order_scalar[col] = torch.nn.Parameter(torch.nn.init.xavier_uniform_(torch.empty(1, 1)))

        # second order feature interactions
        self.second_order_embd = torch.nn.ModuleDict({})
        for col in self.emb_columns:
            self.second_order_embd[col] = torch.nn.Embedding(nrof_cat[col], emb_dim)
        self.second_order_scalar = torch.nn.ParameterDict({})
        for col in numeric_columns:
            self.second_order_scalar[col] = torch.nn.Parameter(torch.nn.init.xavier_uniform_(torch.empty(emb_dim, 1)))

    def forward(self, input_data):
        numeric_features = torch.stack([input_data[col] for col in self.numeric_columns], dim=1)
        numeric_features = self.numeric_feature_bn(numeric_features)

        # first order feature interactions
        # categorical_columns
        first_order_embd_output = None
        for col in self.emb_columns:
            if first_order_embd_output is None:
                first_order_embd_output = self.first_order_embd[col](input_data[col].long())
            else:
                first_order_embd_output = torch.cat([first_order_embd_output, self.first_order_embd[col](input_data[col].long())], dim=1)

        # numeric_columns
        first_order_embd_output = torch.squeeze(first_order_embd_output, dim=2)
        for i, col in enumerate(self.numeric_columns):
            if first_order_embd_output is None:
                first_order_embd_output = torch.mul(numeric_features[i], self.first_order_scalar[col])
            else:
                first_order_embd_output = torch.cat(
                    [first_order_embd_output, torch.mul(numeric_features[:, i], self.first_order_scalar[col])], dim=1)

        # second order feature interactions
        # categorical_columns
        second_order_embd_output = None
        for i, col in enumerate(self.emb_columns):
            if second_order_embd_output is None:
                second_order_embd_output = self.second_order_embd[col](input_data[col].long())
            else:
                second_order_embd_output = torch.cat([second_order_embd_output, self.second_order_embd[col](input_data[col].long())], dim=1)

        # numeric_columns
        for i, col in enumerate(self.numeric_columns):
            if second_order_embd_output is None:
                second_order_embd_output = torch.mul(numeric_features[i], self.second_order_scalar[col])
            else:
                second_order_embd_output = torch.cat(
                    [second_order_embd_output, torch.unsqueeze(torch.mul(
                        numeric_features[:, i], torch.squeeze(
                            torch.stack([self.second_order_scalar[col]] * len(numeric_features)), 2)), 1)], dim=1)

        first_order_embd_output = self.first_feature_bn(first_order_embd_output)
        second_order_embd_output = self.second_feature_bn(second_order_embd_output)

        return first_order_embd_output, second_order_embd_output

In [5]:
class FMLayer(torch.nn.Module):
    def __init__(self) -> None:
        super().__init__()

    def forward(self, first_order_embd, second_order_embd):
        # sum_square part
        summed_features_embd = torch.sum(second_order_embd, dim=1)
        summed_features_embd_square = torch.square(summed_features_embd)

        # square_sum part
        squared_features_embd = torch.square(second_order_embd)
        squared_sum_features_embd = torch.sum(squared_features_embd, dim=1)

        # second order
        second_order = 0.5 * torch.sub(summed_features_embd_square, squared_sum_features_embd)

        return first_order_embd, second_order

In [6]:
class MLPLayer(torch.nn.Module):
    def __init__(
        self,
        input_size: int,
        nrof_layers: int,
        nrof_neurons: int,
        output_size: int,
    ) -> None:
        super().__init__()

        all_input_sizes = [input_size] + [nrof_neurons] * (nrof_layers - 1)
        layers = []
        for i in range(nrof_layers - 1):
            layers.extend([
                torch.nn.Linear(all_input_sizes[i], nrof_neurons),
                torch.nn.BatchNorm1d(nrof_neurons),
                torch.nn.ReLU(),
            ])

        self.deep_block = torch.nn.Sequential(*layers)
        self.output_layer = torch.nn.Linear(nrof_neurons, output_size)

    def init_weights(self, m):
        if type(m) == torch.nn.Linear:
            torch.nn.init.xavier_uniform(m.weight)
            # m.bias.data.fill_(0.001)

    def forward(self, input_data):
        output = self.deep_block(input_data)
        output = self.output_layer(output)
        return output

In [7]:
class DeepFMNet(torch.nn.Module):
    def __init__(
        self,
        nrof_cat: int,
        emb_dim: int,
        emb_columns: list[str],
        numeric_columns: list[str],
        nrof_layers: int,
        nrof_neurons: int,
        output_size: int,
        nrof_out_classes: int,
    ) -> None:
        super().__init__()

        self.emb_dim = emb_dim
        self.emb_columns = emb_columns
        self.numeric_columns = numeric_columns

        self.features_embd = DenseFeatureLayer(nrof_cat, emb_dim, emb_columns, numeric_columns)
        self.FM = FMLayer()

        input_size = (len(emb_columns) + len(numeric_columns)) * emb_dim
        self.MLP = MLPLayer(input_size, nrof_layers, nrof_neurons, output_size)

        input_size = len(emb_columns) + len(numeric_columns) + emb_dim + output_size
        self.dense_layer = torch.nn.Linear(input_size, nrof_out_classes)

    def forward(self, input_data):
        first_order_embd, second_order_embd = self.features_embd(input_data)
        FM_first_order, FM_second_order = self.FM(first_order_embd, second_order_embd)

        second_order_embd = torch.reshape(second_order_embd, [-1, (len(self.emb_columns) + len(self.numeric_columns)) * self.emb_dim])
        Deep = self.MLP(second_order_embd)

        concat_output = torch.cat([FM_first_order, FM_second_order, Deep], dim=1)
        output = self.dense_layer(concat_output)
        output = torch.squeeze(output, 1)

        return output

In [8]:
EPOCHS = 10
EMBEDDING_SIZE = 5
BATCH_SIZE = 512
NROF_LAYERS = 3
NROF_NEURONS = 50
DEEP_OUTPUT_SIZE = 50
NROF_OUT_CLASSES = 1
LEARNING_RATE = 3e-4
TRAIN_PATH = "train_adult.parquet"
VALID_PATH = "valid_adult.parquet"

In [9]:
class DeepFM:
    def __init__(self) -> None:
        self.train_dataset = CustomDataset(TRAIN_PATH)
        self.train_loader = DataLoader(
            dataset=self.train_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
        )
        self.build_model()
        self.log_params()
        self.train_writer = SummaryWriter("./logs/train")
        self.valid_writer = SummaryWriter("./logs/valid")

    def build_model(self) -> None:
        self.network = DeepFMNet(
            nrof_cat=self.train_dataset.nrof_emb_categories,
            emb_dim=EMBEDDING_SIZE,
            emb_columns=self.train_dataset.embedding_columns,
            numeric_columns=self.train_dataset.numeric_columns,
            nrof_layers=NROF_LAYERS, nrof_neurons=NROF_NEURONS,
            output_size=DEEP_OUTPUT_SIZE,
            nrof_out_classes=NROF_OUT_CLASSES,
        )
        self.loss = torch.nn.BCEWithLogitsLoss()
        self.accuracy = Accuracy(task="binary")
        self.optimizer = optim.Adam(self.network.parameters(), lr=LEARNING_RATE)

    def log_params(self):
        return

    def run_train(self) -> None:
        print("Run train ...")
        step = 0
        for epoch in range(1, EPOCHS + 1):
            self.network.train()
            for features, label in self.train_loader:
                # Reset gradients
                self.optimizer.zero_grad()
                output = self.network(features)

                # Calculate error and backpropagate
                loss = self.loss(output, label)
                output = torch.sigmoid(output)
                loss.backward()
                acc = self.accuracy(output, label.long()).item()

                # Update weights with gradients
                self.optimizer.step()
                self.train_writer.add_scalar("CrossEntropyLoss", loss, step)
                self.train_writer.add_scalar("Accuracy", acc, step)
                step += 1

                if step % 50 == 0:
                    print(f"EPOCH {epoch} STEP {step} : train_loss: {loss.item()} train_acc: {acc}")

In [10]:
deep_fm = DeepFM()
deep_fm.run_train()

Run train ...
EPOCH 1 STEP 50 : train_loss: nan train_acc: 0.74609375
EPOCH 2 STEP 100 : train_loss: nan train_acc: 0.775390625
EPOCH 3 STEP 150 : train_loss: nan train_acc: 0.759765625
EPOCH 4 STEP 200 : train_loss: nan train_acc: 0.765625
EPOCH 5 STEP 250 : train_loss: nan train_acc: 0.765625
EPOCH 6 STEP 300 : train_loss: nan train_acc: 0.763671875
EPOCH 7 STEP 350 : train_loss: nan train_acc: 0.779296875
EPOCH 8 STEP 400 : train_loss: nan train_acc: 0.767578125
EPOCH 9 STEP 450 : train_loss: nan train_acc: 0.734375
EPOCH 10 STEP 500 : train_loss: nan train_acc: 0.736328125
